# 第 75 课：推理缓存命中率、调度负载与容量压测

**从「一次前缀能命中」走向「一整段到达流量能否服务」。**

本课把第74课的**真实CPU因果Transformer、LoRA、完整块KV池与可信隔离的前缀缓存**接到一个可复现的离散事件调度器上。观察在相同负载下：缓存命中分母、实际prefill计算量、队列拒绝、容量等待、各租户完成率，以及所有终态的KV回收。

**本课不使用前课文件。** 全部实现嵌入Notebook，配套脚本也自包含。PyTorch和标准库即可，默认CPU、全离线、不会下载任何模型。

**边界优先：** CPU上真实计算并核对KV，但调度用的是自定义**逻辑tick**，不能把结果写成真实GPU吞吐、TTFT/TPOT或生产服务性能。

## 0. 学习路线与使用说明

先理解四个命中率分母，再看固定负载和容量预检。后半部分实际执行**9组对照**：缓存关/开、严格FIFO/租户轮转、小/大缓存、物理页压力、租户与模型隔离，以及主动取消与超时。

最后做三道默认关闭的独立练习：命中率、P95经验分位数、物理块准入预算。参考答案使用独立函数，不覆盖你的实现。

运行建议：从全新内核「重启并运行全部」。不要求有GPU。若只研究数学，可把靠后的 `RUN_COMPLETE_DEMO` 改为 `False`。训练以外的生成均是**冻结小模型的推理**，不会使用现实用户数据。

脚本命令：

```bash
python lesson_75_cache_hit_scheduler_capacity_stress.py --checks-only
python lesson_75_cache_hit_scheduler_capacity_stress.py --demo-only
python lesson_75_cache_hit_scheduler_capacity_stress.py
python lesson_75_cache_hit_scheduler_capacity_stress.py --reload-run lesson_75_outputs/某次实际运行目录
```

指定的新运行目录不会覆盖其他目录；它不是缓存进程的热重启检查点。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第75课：推理缓存命中率、调度负载与容量压测。

独立离线PyTorch教学实验。真实微型因果Transformer/LoRA在物理KV张量里增量读写，
与全上下文前向核对；在真实已计算KV页上实现完整块前缀复用。
没有PagedAttention融合GPU内核、真实GPU性能数据或生产级服务并发。
"""
from __future__ import annotations

# %% 01 环境、确定性身份与范围
import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter, OrderedDict
import secrets
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson75-load-cache-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "vllm_prefix_cache": "https://docs.vllm.ai/en/latest/design/prefix_caching/",
    "vllm_security": "https://docs.vllm.ai/en/stable/usage/security/",
    "vllm_internals": "https://vllm.ai/blog/2025-09-05-anatomy-of-vllm",
    "vllm_metrics": "https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md",
    "vllm_benchmark": "https://github.com/vllm-project/vllm/blob/main/docs/benchmarking/cli.md",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(7501)
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__,'GPU可用:',torch.cuda.is_available())
print('本课CPU运行，并使用合成请求。')

Python: 3.13.5
PyTorch: 2.10.0+cpu GPU可用: False
本课CPU运行，并使用合成请求。


## 1. 命中率不是一个数字

缓存命中率至少有几种不同分母，报告时必须说明计算对象：

| 指标 | 分子 | 分母 |
|---|---|---|
| 有命中的请求占比 | 至少命中一个完整前缀块的请求数 | 已入场请求数 |
| **前缀token命中率** | 实际复用prompt token数 | **已入场**请求的全部prompt token数 |
| 完整块可缓存token命中率 | 实际复用的token数 | 已入场prompt可形成完整块的token数 |
| 前缀块查询命中率 | 命中的完整块数 | 已入场prompt的完整块查询数 |

`cache_limit=0`时，上述命中数为0。空分母应表示为 `None`/N/A，不要伪称0%。负载大时，**拒绝入场的请求不进入prefill命中率分母**；否则仅靠拒绝难题就可以人为改写命中率。

vLLM公开的指标也强调累积计数与时间窗口中的命中比例，监控实现必须区分统计口径。[vLLM metrics说明](https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md)

请先手算：总prompt 100 tokens、其中88 tokens命中。请求占比未必等于88%，因为每条请求长短不同。

In [3]:
print('token命中率手算:',88/100)
print('课后需能区分：请求命中占比、原始token占比与完整块可缓存token占比。')

token命中率手算: 0.88
课后需能区分：请求命中占比、原始token占比与完整块可缓存token占比。


## 2. 真实KV的计算路径（复习第73、74课）

这不是「只在字典里加一个hit计数」的缓存模拟。每个KV物理页实际保存：

```text
[block, layer, key_or_value, slot, head, head_dim]
```

每个新token要写入各层K/V，用历史K/V做一次因果注意力；命中完整前缀块时复用**已经算过的K/V和块末端logits**，然后只计算后续token。`assert_consistent`会与同权重模型的**全上下文前向**比对。

物理页引用计数同时包含活动会话和缓存pin；关闭会话不一定释放仍被cache pin持有的页。内存不足可逐出闲置缓存pin，不得逐出仍有活动引用的页。流量结束后必须移除所有pin、释放会话并清零张量。

下面是完整独立的模型/真实KV实现。为了突出负载思维，将较长的基础实现单元默认折叠，仍可展开阅读。

In [4]:
# %% 02 固定教学分词器与真实因果/LoRA模型（独立于前课文件）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [5]:
# %% 03 同基座不同Adapter的只读模型版本
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [6]:
# %% 04 实体KV页池：容量、块表、引用计数与回滚
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

In [7]:
# %% 05 增量模型执行器：Q/K/V真实写入、块表聚合与全上下文比较
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

## 3. 什么才算“同一个可复用前缀”？

块哈希链把上游父块、当前块token与位置，绑定到：权重版本、LoRA、tokenizer、模板、可信租户共享域与媒体身份。

不同租户即使文字完全相同，也不能凭字符串相等获得缓存；可信授权范围内的多个工作者可以共享。更换LoRA或模板必须失效。只有**完整块**可复用，尾部不满块不公开缓存。

这里的 principal 来自**本地可信调用约定**，不构成真实HTTP鉴权。盐在服务端随机生成，不进入用户请求或原始审计文件。缓存损坏必须明确拒绝，不能当作普通miss掩盖。

具体逻辑来自上一课，但在本Notebook完整复制，不依赖外部文件。[vLLM Prefix Caching](https://docs.vllm.ai/en/latest/design/prefix_caching/) · [vLLM Security](https://docs.vllm.ai/en/stable/usage/security/)

In [8]:
# %% 06 可信共享范围、模型/模板/媒体身份与完整块哈希链
@dataclass(frozen=True)
class CacheRequest:
    """教学内部调用：principal必须由可信网关鉴权后传入，不能让客户端自行声明。"""
    sid: str
    principal: str
    version: str
    messages: tuple[tuple[str,str], ...]
    template_revision: str = "chat-v1"
    media_bytes: bytes | None = None
    add_generation_prompt: bool = True


class TrustedScopeAuthority:
    """服务器控制隔离组与加密随机盐，不接受请求自行提供salt。

    本地示例没有HTTP/登录鉴权。真实调用者必须由网关先确认principal身份，
    本类不防御攻击者能直接调用内部Python对象的情形。
    """
    def __init__(self, principal_groups: dict[str,str]):
        require(type(principal_groups) is dict and bool(principal_groups), '需要非空可信身份组')
        require(all(type(k) is str and k and type(v) is str and v
                    for k,v in principal_groups.items()),'可信身份/共享组非法')
        self._groups=principal_groups.copy()
        self._secrets={g:secrets.token_bytes(32) for g in set(principal_groups.values())}

    def group(self,principal: str) -> str:
        require(type(principal) is str and principal in self._groups,
                '未知或未认证主体不能使用Prefix Cache')
        return self._groups[principal]

    def fingerprint(self,principal: str) -> str:
        group=self.group(principal)
        return hashlib.sha256(self._secrets[group]).hexdigest()

    def rotate(self,group: str) -> str:
        require(group in self._secrets, '未知共享组')
        old=hashlib.sha256(self._secrets[group]).hexdigest()
        self._secrets[group]=secrets.token_bytes(32)
        require(self.fingerprint(next(k for k,v in self._groups.items() if v==group))!=old,
                '加密随机盐轮换未生效')
        return old

    def shared(self,p1:str,p2:str)->bool:
        return self.group(p1)==self.group(p2)


@dataclass(frozen=True)
class CacheIdentity:
    version: str
    model_hash: str
    base_hash: str
    tokenizer_hash: str
    template_hash: str
    scope_hash: str
    media_hash: str
    compute_hash: str


def request_tokens(tok:ByteMergeChatTokenizer,req:CacheRequest)->list[int]:
    require(isinstance(req,CacheRequest),'必须传入CacheRequest')
    require(type(req.messages) is tuple and bool(req.messages),'消息必须为非空tuple')
    require(type(req.add_generation_prompt) is bool,'生成提示选项非法')
    bounded_str(req.template_revision,64,'模板修订号')
    require(bool(req.template_revision),'模板修订号不能为空')
    messages=[]
    for item in req.messages:
        require(type(item) is tuple and len(item)==2 and item[0] in tok.role_ids
                and type(item[1]) is str,'消息必须为(role,content)')
        messages.append({'role':item[0],'content':item[1]})
    return tok.apply_chat_template(messages,add_generation_prompt=req.add_generation_prompt)


def make_identity(req:CacheRequest,authority:TrustedScopeAuthority,tok:ByteMergeChatTokenizer,
                  models:dict[str,TinyChatLM],saved_hashes:dict[str,str]) -> CacheIdentity:
    require(req.version in models and req.version in saved_hashes,'版本没有被可信模型库登记')
    authority.group(req.principal)
    require(type(req.media_bytes) in (bytes,type(None)),'媒体身份必须取自受信预处理的原始字节')
    require(req.media_bytes is None or len(req.media_bytes)<=4096,
            '教学元数据媒体上限4096字节；不允许超大请求键')
    # 禁止在同一个版本名称下原地修改模型而继续读取旧KV。
    model=models[req.version]
    current=state_hash(model.state_dict())
    require(current==saved_hashes[req.version], '热更新必须显式替换并失效缓存，不能原地修改模型')
    return CacheIdentity(
        version=req.version,
        model_hash=current,
        base_hash=base_hash(model),
        tokenizer_hash=digest(tok.contract()),
        template_hash=digest({'family':'toy-message-boundary','revision':req.template_revision,
                              'generation_prompt':req.add_generation_prompt,
                              'role_end':'assistant=eos,others=im_end',
                              'tokenizer':tok.contract()}),
        scope_hash=authority.fingerprint(req.principal),
        media_hash=('none' if req.media_bytes is None else
                    hashlib.sha256(req.media_bytes).hexdigest()),
        compute_hash=digest({'cfg':asdict(model.cfg),'dtype':str(next(model.parameters()).dtype),
                             'position':'absolute','attention':'full causal',
                             'kv_layout':'blocks,layer,kv,slot,heads,dim',
                             'runtime':'toy-cpu-fp32-v1'}),
    )


def prefix_keys(identity:CacheIdentity,ids:list[int],block_size:int)->list[str]:
    require(isinstance(identity,CacheIdentity),'缺少可信缓存身份')
    require(type(block_size) is int and block_size>0,'block_size必须为正整数')
    require(type(ids) is list and all(type(t) is int and t>=0 for t in ids),'token列表非法')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    keys=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        block=ids[start:start+block_size]
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':block})
        keys.append(parent)
    return keys


@dataclass
class CachedBlock:
    key: str
    physical_page: int
    identity: CacheIdentity
    prefix_tokens: tuple[int,...]   # 仅在可信进程内保留，用于哈希碰撞检测；报告禁止展示
    block_tokens: tuple[int,...]
    parent_key: str | None
    end_logits: torch.Tensor
    kv_hash: str
    logits_hash: str
    last_access_tick: int

In [9]:
# %% 07 对真实KV页加缓存所有权：命中、LRU、失效、物理回收
class PrefixKVPool(PagedKVPool):
    """缓存pin与活动会话引用共同拥有物理页。只发布已提交的完整块。

    与工业vLLM不同：CPU同步、预分配Tensor、无真实异步锁、无融合页注意力核。
    精确前缀tokens用于碰撞检测，存储成本未计入KV字节预算。
    """
    def __init__(self,capacity_blocks:int,block_size:int,model_cfg:ToyConfig,versions:set[str],
                 cache_capacity_blocks:int|None=None):
        super().__init__(capacity_blocks,block_size,model_cfg,versions)
        cache_capacity_blocks=capacity_blocks if cache_capacity_blocks is None else cache_capacity_blocks
        require(type(cache_capacity_blocks) is int and 0<=cache_capacity_blocks<=capacity_blocks,
                '缓存块上限不能超过物理容量')
        self.cache_capacity_blocks=cache_capacity_blocks
        self.cache:dict[str,CachedBlock]={}
        self.clock=0
        self.hit_blocks=0;self.hit_tokens=0;self.miss_requests=0
        self.promotions=0;self.evictions=0;self.invalidations=0
        self.skipped_promotions=0;self.zero_hit_requests=0

    def _tick(self):
        self.clock+=1
        return self.clock

    def _page_hash(self,bid:int)->str:
        return state_hash({'physical_KV':self.data[bid]})

    def _validate_entry(self,e:CachedBlock,identity:CacheIdentity,desired_prefix:tuple[int,...],
                        requested_key:str):
        require(e.key==requested_key and e.identity==identity and
                e.prefix_tokens==desired_prefix,'哈希碰撞或缓存身份/前缀不一致，必须拒绝命中')
        require(e.block_tokens==desired_prefix[-self.block_size:]
                and len(e.block_tokens)==self.block_size,'只能命中完整块')
        require(e.kv_hash==self._page_hash(e.physical_page),'缓存KV字节损坏，拒绝复用')
        require(e.logits_hash==state_hash({'end_logits':e.end_logits}),
                '缓存块末端logits字节损坏，拒绝复用')
        require(self.refs[e.physical_page]>=1,'缓存页已经释放，索引悬空')

    def lookup(self,identity:CacheIdentity,ids:list[int])->list[CachedBlock]:
        keys=prefix_keys(identity,ids,self.block_size)
        found=[]
        for i,key in enumerate(keys):
            e=self.cache.get(key)
            if e is None:break
            self._validate_entry(e,identity,tuple(ids[:(i+1)*self.block_size]),key)
            # 顺序访问同时更新LRU；匹配只返回连续前缀，不能跳过中间缺失块。
            e.last_access_tick=self._tick()
            found.append(e)
        return found

    def _evict_one_idle(self)->bool:
        choices=[e for e in self.cache.values() if self.refs[e.physical_page]==1]
        if not choices:return False
        victim=min(choices,key=lambda e:(e.last_access_tick,e.key))
        del self.cache[victim.key]
        self._decref(victim.physical_page)
        self.evictions+=1
        return True

    def _alloc(self):
        if not self.free:
            self._evict_one_idle()
        return super()._alloc()

    def begin(self,sid,token_id,max_length):
        # 父类在发现free为空时会先报OOM；尝试驱逐idle cache才进入它的begin。
        if sid in self.sessions:
            s=self.sessions[sid]
            if s.pending is None and not self.free:
                slot=len(s.tokens)%self.block_size
                needs_page=slot==0 or (slot>0 and self.refs[s.block_table[-1]]>1)
                if needs_page:self._evict_one_idle()
        return super().begin(sid,token_id,max_length)

    def publish(self,sid:str,index:int,key:str,identity:CacheIdentity,
                prefix_tokens:tuple[int,...],end_logits:torch.Tensor,
                parent_key:str|None)->bool:
        require(sid in self.sessions,'发布会话不存在')
        s=self.sessions[sid]
        require(s.pending is None and s.version==identity.version,'不允许发布未提交/跨版本KV')
        require(type(index) is int and 0<=index<len(s.block_table),'发布块下标错误')
        require(len(prefix_tokens)==(index+1)*self.block_size and
                tuple(s.tokens[:len(prefix_tokens)])==prefix_tokens,'前缀token身份错误')
        require(end_logits.shape==(self.cfg.vocab_size,) and end_logits.dtype==self.dtype and
                bool(torch.isfinite(end_logits).all()),'末端logits形状/数值错误')
        chain=prefix_keys(identity,list(prefix_tokens),self.block_size)
        require(chain and chain[-1]==key and (chain[-2] if len(chain)>1 else None)==parent_key,
                '发布哈希链与token前缀不一致')
        if key in self.cache:
            self._validate_entry(self.cache[key],identity,prefix_tokens,key)
            return False
        if self.cache_capacity_blocks==0:
            self.skipped_promotions+=1
            return False
        if len(self.cache)>=self.cache_capacity_blocks and not self._evict_one_idle():
            self.skipped_promotions+=1
            return False
        bid=s.block_table[index]
        require(self.refs[bid]>=1,'发布物理页没有会话所有权')
        self.refs[bid]+=1             # 独立cache pin：请求结束后仍可命中
        logit=end_logits.detach().cpu().contiguous().clone()
        self.cache[key]=CachedBlock(key=key,physical_page=bid,identity=identity,
             prefix_tokens=prefix_tokens,block_tokens=prefix_tokens[-self.block_size:],
             parent_key=parent_key,end_logits=logit,kv_hash=self._page_hash(bid),
             logits_hash=state_hash({'end_logits':logit}),last_access_tick=self._tick())
        self.promotions+=1
        self.audit()
        return True

    def invalidate(self,predicate:Callable[[CachedBlock],bool]|None=None)->int:
        """移除pin。即使仍有活动会话，只有它们的引用会保留KV。"""
        selected=[key for key,e in self.cache.items() if predicate is None or predicate(e)]
        for key in selected:
            e=self.cache.pop(key)
            self._decref(e.physical_page)
        self.invalidations+=len(selected)
        self.audit()
        return len(selected)

    def advance_clock_and_expire(self,ticks:int,max_idle_ticks:int)->int:
        require(type(ticks) is int and ticks>=0 and type(max_idle_ticks) is int and max_idle_ticks>=0,
                '逻辑时间参数非法')
        self.clock+=ticks
        return self.invalidate(lambda e: self.clock-e.last_access_tick>max_idle_ticks)

    def cache_usage(self)->dict:
        total=self.usage()
        idle_cached=sum(self.refs[e.physical_page]==1 for e in self.cache.values())
        return {'cache_blocks':len(self.cache),'cache_capacity_blocks':self.cache_capacity_blocks,
                'idle_cached_blocks':idle_cached,'active_cached_blocks':len(self.cache)-idle_cached,
                'cache_pins':len(self.cache),'cache_held_bytes':len(self.cache)*self.bytes_per_block,
                'hit_blocks':self.hit_blocks,'hit_tokens':self.hit_tokens,
                'promotions':self.promotions,'evictions':self.evictions,
                'invalidations':self.invalidations,'skipped_promotions':self.skipped_promotions,
                'live_blocks':total['live_blocks'],'free_blocks':total['free_blocks'],
                'session_count':len(self.sessions),'physical_storage_bytes':total['physical_storage_bytes']}

    def usage(self)->dict:
        result=super().usage()
        result['cache_pins']=len(self.cache)
        result['cache_held_bytes']=len(self.cache)*self.bytes_per_block
        # 物理页可以只由缓存持有，因此旧课的logical_refs-live会变负：需把pin计入。
        result['saved_physical_blocks_by_sharing']=(result['logical_page_references']+
                                                     len(self.cache)-result['live_blocks'])
        return result

    def audit(self)->dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话模型版本不在白名单')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '会话页数与token长度不匹配')
            require(len(s.block_table)==len(set(s.block_table)),'同一会话重复引用页')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'页ID越界')
                expected[bid]+=1
        cache_pages=set()
        for key,e in self.cache.items():
            require(key==e.key and e.physical_page not in cache_pages,'重复pin或缓存键/页错配')
            cache_pages.add(e.physical_page)
            require(0<=e.physical_page<self.capacity_blocks,'缓存物理页非法')
            require(len(e.prefix_tokens)%self.block_size==0 and len(e.block_tokens)==self.block_size,
                    '缓存只允许完整块')
            expected[e.physical_page]+=1
        require(expected==self.refs,'会话引用+缓存pin不等于实际物理引用数')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free-list不守恒')
        require(len(set(self.free))==len(self.free),'free-list有重复')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),'空闲页未清零')
        u=self.usage()
        require(u['saved_physical_blocks_by_sharing']>=0,'物理共享账本负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '字节账本不守恒')
        return u

In [10]:
# %% 08 前缀命中后继续真实预填充，并与完整因果前向核对
class PrefixCacheRunner:
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],
                 pool:PrefixKVPool,authority:TrustedScopeAuthority):
        require(isinstance(pool,PrefixKVPool),'需要有缓存所有权的KV页池')
        self.tok=tok;self.models=models;self.pool=pool;self.authority=authority
        self.runner=CachedModelRunner(tok,models,pool)
        self.bound:dict[str,CacheIdentity]={}
        self.events:list[dict]=[]

    def admit(self,req:CacheRequest)->dict:
        require(isinstance(req,CacheRequest),'请求类型非法')
        self.pool._sid(req.sid)
        require(req.sid not in self.pool.sessions,'sid不可重复入场')
        # 注意：principal须来自已鉴权的内部调用上下文，不应接受HTTP客户端自报。
        ids=request_tokens(self.tok,req)
        require(bool(ids) and len(ids)<=self.pool.cfg.max_length,'提示过长或为空')
        ident=make_identity(req,self.authority,self.tok,self.models,self.runner.model_hashes)
        hits=self.pool.lookup(ident,ids)
        hit_count=len(hits)*self.pool.block_size
        keys=prefix_keys(ident,ids,self.pool.block_size)
        self.pool.open(req.sid,req.version)
        self.bound[req.sid]=ident
        end_logits:dict[int,torch.Tensor]={}
        try:
            state=self.pool.sessions[req.sid]
            for i,e in enumerate(hits):
                self.pool.refs[e.physical_page]+=1
                state.block_table.append(e.physical_page)
                state.tokens.extend(e.block_tokens)
                end_logits[i]=e.end_logits.detach().clone()
            if hits:
                self.runner.last_logits[req.sid]=hits[-1].end_logits.detach().clone()
            self.pool.audit()
            for t in ids[hit_count:]:
                out=self.runner.step(req.sid,t)
                n=len(self.pool.sessions[req.sid].tokens)
                if n%self.pool.block_size==0:
                    end_logits[n//self.pool.block_size-1]=out.detach().clone()
            require(self.pool.sessions[req.sid].tokens==ids,'前缀命中恢复后的token序列错误')
            require(req.sid in self.runner.last_logits,'请求没有任何合法预测logits')
            # 先计算整个prompt，再发布完整块；失败入场不残留不完整页索引。
            parent=None
            for i,key in enumerate(keys):
                prefix=tuple(ids[:(i+1)*self.pool.block_size])
                require(i in end_logits,'完整块末端logits没有计算/回填')
                self.pool.publish(req.sid,i,key,ident,prefix,end_logits[i],parent)
                parent=key
            self.pool.hit_blocks+=len(hits);self.pool.hit_tokens+=hit_count
            if not hits:self.pool.zero_hit_requests+=1
            max_delta=self.runner.assert_consistent(req.sid)
            result={'sid':req.sid,'version':req.version,'prompt_tokens':len(ids),
                    'hit_blocks':len(hits),'hit_tokens':hit_count,'new_tokens_computed':len(ids)-hit_count,
                    'max_logits_abs_delta':max_delta,'cache_blocks_after':len(self.pool.cache),
                    'scope':'internal_authenticated_principal'}
            self.events.append(result.copy())
            return result
        except Exception:
            # 失败不留下活动会话/泄漏物理页；不回滚之前已发生的idle LRU驱逐。
            if req.sid in self.pool.sessions:
                self.runner.close(req.sid)
            self.bound.pop(req.sid,None)
            self.pool.audit()
            raise

    def continue_token(self,sid:str,token_id:int)->float:
        require(sid in self.bound and sid in self.pool.sessions,'不存在的会话')
        identity=self.bound[sid]
        require(identity.model_hash==state_hash(self.models[identity.version].state_dict()),
                '活跃会话期间模型权重不可原地热修改')
        self.runner.step(sid,token_id)
        return self.runner.assert_consistent(sid)

    def fork_session(self,parent:str,child:str,principal:str):
        require(parent in self.bound and parent in self.pool.sessions,'父会话不存在')
        require(child not in self.bound and child not in self.pool.sessions,'子会话已经存在')
        identity=self.bound[parent]
        require(self.authority.fingerprint(principal)==identity.scope_hash,
                'fork只允许相同可信缓存共享范围，不能跨租户')
        self.runner.fork(parent,child,identity.version)
        self.bound[child]=identity
        self.pool.audit()

    def close(self,sid:str):
        require(sid in self.bound,'未授权/已经关闭的会话')
        self.runner.close(sid)
        self.bound.pop(sid)
        self.pool.audit()

    def invalidate_version(self,version:str)->int:
        require(version in self.models,'未知版本')
        return self.pool.invalidate(lambda e:e.identity.version==version)

    def rotate_group(self,group:str)->int:
        old_fp=self.authority.rotate(group)
        return self.pool.invalidate(lambda e:e.identity.scope_hash==old_fp)

    def replace_version(self,version:str,new_model:TinyChatLM)->int:
        require(version in self.models,'未知版本不能替换')
        require(all(s.version!=version for s in self.pool.sessions.values()),
                '有该版本的活跃会话，禁止原地替换/撕裂读写')
        require(isinstance(new_model,TinyChatLM) and new_model.cfg==self.pool.cfg,
                '新模型配置或类型错误')
        require(next(new_model.parameters()).device.type=='cpu'
                and next(new_model.parameters()).dtype==torch.float32,
                '新模型必须是CPU FP32')
        require(all(not p.requires_grad for p in new_model.parameters()),'新模型必须全部冻结')
        replacement=copy.deepcopy(new_model).eval()
        invalidated=self.invalidate_version(version)
        self.models[version]=replacement
        self.runner.models[version]=replacement
        self.runner.model_hashes[version]=state_hash(replacement.state_dict())
        return invalidated

    def final_flush(self)->dict:
        require(not self.pool.sessions and not self.bound,'flush前必须结束全部会话')
        self.pool.invalidate()
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and not bool(self.pool.data.count_nonzero()),
                '全部缓存清空后仍有KV残留')
        return usage

In [11]:
tok,models=build_models(7501)
print('版本:',sorted(models))
print('模型参数量:',sum(p.numel() for p in models['v0001'].parameters()))
print('版本确实不同:',state_hash(models['v0001'].state_dict())!=state_hash(models['v0002'].state_dict()))
print('模型已冻结:',all(not p.requires_grad for model in models.values() for p in model.parameters()))

版本: ['v0001', 'v0002']
模型参数量: 42048
版本确实不同: True
模型已冻结: True


## 4. 先把负载写成“确定的输入记录”

每条合成请求包含：到达tick、租户、版本、正文、最长等待、最多解码步数和可选主动取消时间。数据集不来自日志或现实聊天。

演练负载分五类：

1. **steady_hot**：间隔较宽、热prompt重复，检验缓存是否减少真实prefill计算。
2. **burst_skew**：到达密集且租户占比不均，检验队列拒绝、超时与调度策略。
3. **thrash**：冷热混合正文，检验小/大缓存pin容量对命中的影响。
4. **isolation**：相同正文、不同租户与LoRA版本，检验跨范围隔离。
5. **cancel**：队列与活动会话主动取消、期限到期、最终KV回收。

**不要把逻辑tick解释为毫秒。** 一个tick能让每个活跃会话最多生成一个token；prefill在本课同一调度tick内同步完成。该抽象不代表GPU Prefill/Decode同时运行的真实时序。

In [12]:
# %% 09 可复现的合成到达序列、容量配置与指标定义
@dataclass(frozen=True)
class TrafficRequest:
    sid: str
    arrival_tick: int
    principal: str
    body: str
    version: str = 'v0001'
    template_revision: str = 'chat-v1'
    patience_ticks: int = 16
    decode_steps: int = 2
    cancel_after_ticks: int | None = None

    def validate(self):
        bounded_str(self.sid, 48, 'sid')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+', self.sid)), 'sid字符非法')
        bounded_str(self.body, 512, '请求正文')
        require(self.principal in {'alice','bob','alice_worker','org1','org2'}, '请求主体必须来自可信测试名单')
        require(self.version in {'v0001','v0002'}, '未知模型版本')
        require(self.template_revision in {'chat-v1','chat-v2'}, '未知模板修订号')
        require(type(self.arrival_tick) is int and self.arrival_tick >= 0, '到达tick必须非负整数')
        require(type(self.patience_ticks) is int and self.patience_ticks > 0, '等待截止tick必须正整数')
        require(type(self.decode_steps) is int and 1 <= self.decode_steps <= 3, '生成预算1到3步')
        require(self.cancel_after_ticks is None or
                (type(self.cancel_after_ticks) is int and self.cancel_after_ticks >= 1),
                '主动取消必须是到达后的正整数tick')


@dataclass(frozen=True)
class LoadConfig:
    capacity_blocks: int = 64
    cache_limit: int = 24
    block_size: int = 4
    max_active: int = 3
    per_principal_active: int = 2
    queue_limit: int = 12
    max_ticks: int = 180
    policy: str = 'tenant_rr'  # FIFO严格队头，或租户RR跳过不可调度队头

    def validate(self):
        for k in ('capacity_blocks','block_size','max_active','per_principal_active',
                  'queue_limit','max_ticks'):
            v=getattr(self,k)
            require(type(v) is int and v > 0, k+'必须是正整数')
        require(type(self.cache_limit) is int and 0 <= self.cache_limit <= self.capacity_blocks,
                '缓存容量必须在0..物理页容量之间')
        require(self.per_principal_active <= self.max_active, '单租户并行度不能超过总并行度')
        require(self.policy in {'fifo','tenant_rr'}, '只支持严格FIFO/租户轮转')


def make_workload(kind: str, count: int=18) -> list[TrafficRequest]:
    """确定性的教学流量，不含真实用户请求；组内复用与跨租户请求交错。"""
    require(kind in {'steady_hot','burst_skew','isolation','thrash','cancel'}, '未知负载形态')
    require(type(count) is int and 2 <= count <= 120, '请求数量2..120')
    traffic=[]
    for i in range(count):
        if kind=='steady_hot':
            principal='alice' if i%4 else 'alice_worker'  # 可信同组，允许共享缓存
            arrival=3*i
            body='Hello world! '+('repeat' if i%5 else 'alternate')
        elif kind=='burst_skew':
            principal='bob' if i%6==0 else 'alice'
            arrival=(i//6)*2
            body='Hello world! '+('repeat' if i%3 else f'cold{i:02d}')
        elif kind=='isolation':
            principal='alice' if i%2==0 else 'bob'
            arrival=3*i
            body='Hello world! repeat'
        elif kind=='thrash':
            principal='alice'
            arrival=3*i
            body='Hello world! '+('repeat' if i%3 else f'unique-{i:02d}')
        else:
            principal='alice' if i%3 else 'bob'
            arrival=i//4
            body='Hello world! '+('repeat' if i%2 else 'alternate')
        traffic.append(TrafficRequest(sid=f'{kind[:3]}_{i:03d}',arrival_tick=arrival,
                          principal=principal,body=body,
                          version=('v0002' if kind=='isolation' and i%4==3 else 'v0001'),
                          patience_ticks=(5 if kind in {'burst_skew','cancel'} else 18),
                          decode_steps=2,
                          cancel_after_ticks=(2 if kind=='cancel' and i%7==0 else None)))
    validate_workload(traffic)
    return traffic


def validate_workload(items: list[TrafficRequest]) -> None:
    require(type(items) is list and bool(items), '需要非空流量')
    seen=set();last=-1
    for x in items:
        require(isinstance(x,TrafficRequest),'必须是TrafficRequest')
        x.validate()
        require(x.sid not in seen, '请求ID重复')
        require(x.arrival_tick>=last, '到达顺序必须是非降序；调用方显式排序')
        seen.add(x.sid);last=x.arrival_tick


def nearest_rank(data: list[int], p: float) -> int | None:
    """经验分位数nearest-rank：第ceil(p*n)个，空数据返回None。"""
    require(0 < p <= 1 and math.isfinite(p), '分位数p必须(0,1]')
    if not data:return None
    require(all(type(x) is int and x>=0 for x in data), '必须是非负整数的逻辑tick')
    ordered=sorted(data)
    return ordered[math.ceil(p*len(data))-1]


def metric_ratio(numerator: int, denominator: int) -> float | None:
    require(type(numerator) is int and type(denominator) is int and
            0 <= numerator <= denominator, '计数器不符合比例边界')
    return numerator/denominator if denominator else None


def jain_index(values: list[float]) -> float | None:
    """对各主体完成/到达比例算Jain，所有比例0则不定义；不能取代按主体报告。"""
    require(bool(values) and all(isinstance(x,(float,int)) and math.isfinite(x) and x>=0 for x in values),
            'Jain输入需要有限非负数')
    total=sum(values);denom=sum(x*x for x in values)
    return (total*total)/(len(values)*denom) if denom else None

In [13]:
for kind in ['steady_hot','burst_skew','thrash','isolation','cancel']:
    rows=make_workload(kind,8)
    print(kind,'请求数',len(rows),'到达tick',[x.arrival_tick for x in rows[:6]],
          '主体',[x.principal for x in rows[:4]])
print('默认容量配置:',asdict(LoadConfig()))

steady_hot 请求数 8 到达tick [0, 3, 6, 9, 12, 15] 主体 ['alice_worker', 'alice', 'alice', 'alice']
burst_skew 请求数 8 到达tick [0, 0, 0, 0, 0, 0] 主体 ['bob', 'alice', 'alice', 'alice']
thrash 请求数 8 到达tick [0, 3, 6, 9, 12, 15] 主体 ['alice', 'alice', 'alice', 'alice']
isolation 请求数 8 到达tick [0, 3, 6, 9, 12, 15] 主体 ['alice', 'bob', 'alice', 'bob']
cancel 请求数 8 到达tick [0, 0, 0, 0, 1, 1] 主体 ['bob', 'alice', 'alice', 'bob']
默认容量配置: {'capacity_blocks': 64, 'cache_limit': 24, 'block_size': 4, 'max_active': 3, 'per_principal_active': 2, 'queue_limit': 12, 'max_ticks': 180, 'policy': 'tenant_rr'}


## 5. 队列和物理容量是两种不同的压力

**队列满**：新请求在入场之前被明确拒绝，记录 `queue_rejected`；这些请求没有执行prefill。

**活动槽满**：请求等待 `max_active` 释放；不等于KV页耗尽。

**物理页不足**：只读准入预检根据实际可复用页、free页与可逐出的**闲置缓存pin**，同时考虑本请求后续解码和活动请求的尚未使用预算。不能把正在被活动会话引用的页视为可逐出。

**永远装不下**：单个prompt加有限解码预算所需块数大于整个物理容量，直接 `oversize_rejected`，不在队列里无限旋转。

这份预算采用保守预留策略，**不是生产调度器的最佳容量规划**，也未实现抢占/恢复。vLLM的实际调度涉及token budget、KV分配、排队及可能的抢占，不能用这个单线程模型替代。[vLLM scheduler](https://docs.vllm.ai/en/latest/api/vllm/v1/core/sched/scheduler/)

在本课，只读规划不应该修改LRU时钟或偷偷消耗实际KV页；真正的入场会按已有前缀复用并执行新token计算。

In [14]:
# %% 10 物理KV约束下的调度器：明确队列、预检、租户限额、终态与统计
class LoadScheduler:
    """同步CPU按逻辑tick离散推进；一次入场做真实微型模型prefill。

    这不是GPU时间模型：每tick对每个活跃请求至多生成1token，prefill不占tick，
    模型实时计算花费的墙钟时间不写入仿真报告，也不据此声称TTFT/TPOT/goodput。
    """
    def __init__(self,service:PrefixCacheRunner,config:LoadConfig,items:list[TrafficRequest]):
        config.validate();validate_workload(items)
        require(items[-1].arrival_tick < config.max_ticks, '逻辑仿真窗口必须覆盖最后到达的请求')
        require(service.pool.capacity_blocks==config.capacity_blocks and
                service.pool.block_size==config.block_size and
                service.pool.cache_capacity_blocks==config.cache_limit,
                '负载容量配置和物理KV池不符')
        self.service=service;self.pool=service.pool;self.cfg=config
        self.items=items
        self.by_sid={r.sid:r for r in items}
        self.pending:list[str]=[]
        self.active:dict[str,dict]={}
        self.terminal:dict[str,dict]={}
        self.admitted:dict[str,dict]={}
        self.rr_tenants=sorted({r.principal for r in items})
        self.rr_next=0
        self.arrival_cursor=0
        self.capacity_deferrals=0;self.quota_deferrals=0
        self.slot_deferrals=0;self.queue_peak=0;self.peak_active=0
        self.prompts_computed=0;self.prompts_reused=0;self.total_prompt_tokens=0
        self.total_eligible_tokens=0;self.hit_requests=0
        self.peak_live_blocks=0;self.active_time_slots=0
        self.model_errors=0;self.finalized=False
        self.trace:list[dict]=[]
        self.rr_admit_order:list[str]=[]

    def _request(self,sid):
        r=self.by_sid[sid]
        return CacheRequest(sid=r.sid, principal=r.principal, version=r.version,
              messages=(('system','Reply briefly.'),('user',r.body)),
              template_revision=r.template_revision)

    def _cancel_due(self, sid:str, tick:int) -> bool:
        r=self.by_sid[sid]
        return r.cancel_after_ticks is not None and tick-r.arrival_tick>=r.cancel_after_ticks

    def _deadline_due(self,sid:str,tick:int) -> bool:
        return tick-self.by_sid[sid].arrival_tick>=self.by_sid[sid].patience_ticks

    def _finish(self,sid:str, reason:str, tick:int) -> None:
        require(sid not in self.terminal and sid in self.by_sid, '重复终止或未知sid')
        require(reason in {'completed_eos','completed_cap','queue_rejected','oversize_rejected',
                            'cancelled_waiting','cancelled_active','deadline_waiting',
                            'deadline_active','scheduler_timeout','kv_error'}, '终态无效')
        if sid in self.active:
            self.service.close(sid)
            session=self.active.pop(sid)
            decoded=session['decoded']
            first=session['first_token_tick']
        else:
            decoded=0;first=None
        row={'sid':sid,'principal':self.by_sid[sid].principal,'reason':reason,
             'arrival_tick':self.by_sid[sid].arrival_tick,'end_tick':tick,
             'started_tick':self.admitted[sid]['tick'] if sid in self.admitted else None,
             'first_output_tick':first,'decoded_tokens':decoded}
        self.terminal[sid]=row

    def _plan(self, sid:str):
        """只读前缀预检，不更新LRU；缓存损坏必须抛异常，不能转成miss。"""
        req=self._request(sid)
        ids=request_tokens(self.service.tok,req)
        identity=make_identity(req,self.service.authority,self.service.tok,
                              self.service.models,self.service.runner.model_hashes)
        keys=prefix_keys(identity,ids,self.cfg.block_size)
        found=[]
        for i,key in enumerate(keys):
            entry=self.pool.cache.get(key)
            if entry is None:break
            self.pool._validate_entry(entry,identity,tuple(ids[:(i+1)*self.cfg.block_size]),key)
            found.append(entry)
        block_count=math.ceil(len(ids)/self.cfg.block_size)
        max_blocks=math.ceil((len(ids)+self.by_sid[sid].decode_steps-1)/self.cfg.block_size)
        hits=len(found)
        safe_to_evict=sum(e.physical_page not in {f.physical_page for f in found}
                          and self.pool.refs[e.physical_page]==1 for e in self.pool.cache.values())
        # 活动请求未来可能追加token，预留账本并非真实分配；避免无意在decode时OOM。
        reserved=0
        for run_sid,info in self.active.items():
            s=self.pool.sessions[run_sid]
            remaining=self.by_sid[run_sid].decode_steps-info['decoded']
            reserved+=max(0,math.ceil((len(s.tokens)+remaining-1)/self.cfg.block_size)-len(s.block_table))
        available=max(0,len(self.pool.free)+safe_to_evict-reserved)
        return {'ids':ids,'identity':identity,'hit_blocks':hits,
                'hit_keys':[e.key for e in found], 'prompt_blocks':block_count,
                'required_new':max_blocks-hits,'available_new':available,
                'oversize':max_blocks>self.cfg.capacity_blocks}

    def _admit(self,sid,tick:int,plan:dict):
        r=self.by_sid[sid]
        admit=self.service.admit(self._request(sid))
        require(admit['hit_blocks']==plan['hit_blocks'] and
                admit['prompt_tokens']==len(plan['ids']),
                '预检与执行之间的缓存不一致；此单线程教学调度器不允许竞态')
        hit=admit['hit_tokens'];prompt=admit['prompt_tokens']
        self.total_prompt_tokens+=prompt
        self.total_eligible_tokens+=prompt//self.cfg.block_size*self.cfg.block_size
        self.prompts_reused+=hit
        self.prompts_computed+=admit['new_tokens_computed']
        self.hit_requests+=int(hit>0)
        self.admitted[sid]={'tick':tick,'hit_tokens':hit,'prompt_tokens':prompt,
                            'computed_tokens':admit['new_tokens_computed'],
                            'max_logit_delta':admit['max_logits_abs_delta']}
        self.active[sid]={'started_tick':tick,'decoded':0,'first_token_tick':None}
        self.rr_admit_order.append(sid)
        self.pending.remove(sid)
        self.peak_active=max(self.peak_active,len(self.active))

    def _available_tenant(self,principal:str)->bool:
        return sum(self.by_sid[sid].principal==principal for sid in self.active)<self.cfg.per_principal_active

    def _choose(self):
        require(self.pending, '没有排队请求')
        if self.cfg.policy=='fifo':return [self.pending[0]]
        result=[]
        for offset in range(len(self.rr_tenants)):
            tenant=self.rr_tenants[(self.rr_next+offset)%len(self.rr_tenants)]
            match=next((sid for sid in self.pending if self.by_sid[sid].principal==tenant),None)
            if match is not None:result.append(match)
        return result

    def _admit_waiting(self,tick:int):
        while self.pending and len(self.active)<self.cfg.max_active:
            selected=None;plan=None;handled_oversize=False
            for sid in self._choose():
                principal=self.by_sid[sid].principal
                if not self._available_tenant(principal):
                    self.quota_deferrals+=1
                    continue
                proposed=self._plan(sid)
                if proposed['oversize']:
                    self.pending.remove(sid)
                    self._finish(sid,'oversize_rejected',tick)
                    handled_oversize=True
                    break
                if proposed['required_new']>proposed['available_new']:
                    self.capacity_deferrals+=1
                    continue
                selected=sid;plan=proposed
                break
            if handled_oversize:continue
            if selected is None:break
            self._admit(selected,tick,plan)
            if self.cfg.policy=='tenant_rr':
                pos=self.rr_tenants.index(self.by_sid[selected].principal)
                self.rr_next=(pos+1)%len(self.rr_tenants)
        if self.pending and len(self.active)>=self.cfg.max_active:self.slot_deferrals+=1

    def _decode(self,tick:int):
        for sid in list(self.active):
            if self._cancel_due(sid,tick):
                self._finish(sid,'cancelled_active',tick)
                continue
            if self._deadline_due(sid,tick):
                self._finish(sid,'deadline_active',tick)
                continue
            state=self.active[sid]
            if tick<=state['started_tick']:continue
            logits=self.service.runner.last_logits[sid]
            require(logits.shape==(self.service.tok.vocab_size,), '缓存logits形状错误')
            token=int(logits.argmax())
            state['decoded']+=1
            if state['first_token_tick'] is None:state['first_token_tick']=tick
            if token==self.service.tok.eos_token_id:
                self._finish(sid,'completed_eos',tick)
            elif state['decoded']>=self.by_sid[sid].decode_steps:
                self._finish(sid,'completed_cap',tick)
            else:
                try:self.service.continue_token(sid,token)
                except NoKVSpace:
                    self.model_errors+=1
                    self._finish(sid,'kv_error',tick)

    def _step(self,tick:int):
        # 相同tick中，所有请求先到达；若队列已满则明确记录拒绝。
        while self.arrival_cursor<len(self.items) and self.items[self.arrival_cursor].arrival_tick==tick:
            req=self.items[self.arrival_cursor]
            self.arrival_cursor+=1
            if len(self.pending)>=self.cfg.queue_limit:
                self._finish(req.sid,'queue_rejected',tick)
            else:self.pending.append(req.sid)
        self.queue_peak=max(self.queue_peak,len(self.pending))
        for sid in list(self.pending):
            if self._cancel_due(sid,tick):
                self.pending.remove(sid);self._finish(sid,'cancelled_waiting',tick)
            elif self._deadline_due(sid,tick):
                self.pending.remove(sid);self._finish(sid,'deadline_waiting',tick)
        self._decode(tick)
        self._admit_waiting(tick)
        self.pool.audit()
        usage=self.pool.cache_usage()
        self.peak_live_blocks=max(self.peak_live_blocks,usage['live_blocks'])
        self.active_time_slots+=len(self.active)
        self.trace.append({'tick':tick,'arrived':self.arrival_cursor,
                           'queued':len(self.pending),'active':len(self.active),
                           'terminal':len(self.terminal),'physical_live_blocks':usage['live_blocks'],
                           'idle_cached_blocks':usage['idle_cached_blocks'],
                           'cache_pins':usage['cache_pins']})

    def run(self):
        require(not self.finalized, '调度器只能运行一次')
        for tick in range(self.cfg.max_ticks):
            self._step(tick)
            if len(self.terminal)==len(self.items):break
        else:
            tick=self.cfg.max_ticks
            for sid in list(self.pending):
                self.pending.remove(sid);self._finish(sid,'scheduler_timeout',tick)
            for sid in list(self.active):self._finish(sid,'scheduler_timeout',tick)
        require(len(self.terminal)==len(self.items), '到达请求必须逐条进入终态')
        require(self.arrival_cursor==len(self.items), '调度上限必须覆盖全部到达请求')
        require(not self.pending and not self.active and not self.pool.sessions,
                '终态必须释放所有KV会话')
        self.finalized=True
        report=self.report()
        # 刷新最终的cache所有权；不更改已经计算的容量轨迹。
        flush=self.service.final_flush()
        require(flush['live_blocks']==0 and self.pool.data.count_nonzero().item()==0,
                '必须在所有场景后回收并清零物理KV')
        report['final_flush']={'live_blocks':flush['live_blocks'],
                              'cache_pins':flush['cache_pins'],'all_bytes_zero':True}
        return report

    def report(self):
        require(self.finalized,'只有结束全部请求才能汇总统计')
        kinds=Counter(row['reason'] for row in self.terminal.values())
        good={'completed_eos','completed_cap'}
        admitted_count=len(self.admitted)
        completed=sum(v for k,v in kinds.items() if k in good)
        wait=[r['tick']-self.by_sid[sid].arrival_tick for sid,r in self.admitted.items()]
        first=[r['first_output_tick']-r['arrival_tick'] for r in self.terminal.values()
               if r['first_output_tick'] is not None]
        tenants=sorted({r.principal for r in self.items})
        bytenant={}
        for principal in tenants:
            arrivals=sum(x.principal==principal for x in self.items)
            accepted=sum(self.by_sid[x].principal==principal for x in self.admitted)
            outcomes=Counter(r['reason'] for r in self.terminal.values() if r['principal']==principal)
            done=sum(v for k,v in outcomes.items() if k in good)
            bytenant[principal]={'arrived':arrivals,'admitted':accepted,'completed':done,
                 'completion_fraction':done/arrivals,'reasons':dict(sorted(outcomes.items()))}
        prompt_block_queries=sum(rec['prompt_tokens']//self.cfg.block_size for rec in self.admitted.values())
        hit_blocks=self.prompts_reused//self.cfg.block_size
        summary={'requests':len(self.items),'admitted':admitted_count,'completed':completed,
                 'terminal_counts':dict(sorted(kinds.items())),
                 'request_cache_hit_fraction':metric_ratio(self.hit_requests,admitted_count),
                 'prompt_token_hit_fraction':metric_ratio(self.prompts_reused,self.total_prompt_tokens),
                 'eligible_full_block_hit_fraction':metric_ratio(self.prompts_reused,self.total_eligible_tokens),
                 'cache_block_query_hit_fraction':metric_ratio(hit_blocks,prompt_block_queries),
                 'prefill_tokens_total':self.total_prompt_tokens,
                 'prefill_tokens_computed':self.prompts_computed,
                 'prefill_tokens_reused':self.prompts_reused,
                 'admitted_prompt_full_block_queries':prompt_block_queries,
                 'hit_blocks':hit_blocks,
                 'decode_output_tokens':sum(row['decoded_tokens'] for row in self.terminal.values()),
                 'actually_computed_model_tokens':self.service.runner.tokens_processed,
                 'max_logit_abs_delta':max((r['max_logit_delta'] for r in self.admitted.values()),default=0.0),
                 'completion_per_logical_tick':completed/len(self.trace),
                 'logical_ticks':len(self.trace),
                 'queue_peak':self.queue_peak,'active_peak':self.peak_active,
                 'physical_live_blocks_peak':self.peak_live_blocks,
                 'physical_capacity_blocks':self.cfg.capacity_blocks,
                 'kv_pool_bytes_preallocated':self.pool.data.numel()*self.pool.data.element_size(),
                 'kv_bytes_per_block':self.pool.bytes_per_block,
                 'logical_wait_ticks_p50':nearest_rank(wait,.5),
                 'logical_wait_ticks_p95':nearest_rank(wait,.95),
                 'logical_time_to_first_output_ticks_p95':nearest_rank(first,.95),
                 'queue_rejections':kinds['queue_rejected'],
                 'capacity_deferral_events':self.capacity_deferrals,
                 'tenant_quota_deferral_events':self.quota_deferrals,
                 'active_slot_deferral_events':self.slot_deferrals,
                 'kv_errors':self.model_errors,
                 'pool_evictions':self.pool.evictions,
                 'skipped_promotions':self.pool.skipped_promotions,
                 'cache_pins_before_flush':len(self.pool.cache),
                 'fairness_jain_completion_fraction':jain_index([bytenant[t]['completion_fraction'] for t in tenants]),
                 'per_principal':bytenant,
                 'notes':['logical_ticks_not_wallclock','not_TTFT_TPOT_or_GPU_goodput',
                          'cache_hits_are_admitted_prompt_tokens_only',
                          'Jain_index_is_not_fairness_guarantee']}
        require(self.prompts_computed+self.prompts_reused==self.total_prompt_tokens,
                '实际prefill token账本不守恒')
        require(sum(kinds.values())==len(self.items), '终态统计不守恒')
        require(self.peak_active<=self.cfg.max_active and
                self.peak_live_blocks<=self.cfg.capacity_blocks,'超过活动/物理页上限')
        return {'config':asdict(self.cfg),'summary':summary,'admitted':
                  [{**{'sid':sid},**self.admitted[sid]} for sid in sorted(self.admitted)],
                'terminal':[self.terminal[sid] for sid in sorted(self.terminal)],
                'trace':self.trace,'admit_order':self.rr_admit_order}

In [15]:
cfg_probe=LoadConfig(capacity_blocks=48,cache_limit=16,max_active=2)
print('物理块token数:',cfg_probe.block_size)
print('配置最大活动请求数:',cfg_probe.max_active)
print('所有指标都使用逻辑tick，而不把工作项强行换算成毫秒。')

物理块token数: 4
配置最大活动请求数: 2
所有指标都使用逻辑tick，而不把工作项强行换算成毫秒。


## 6. 拒绝、取消、超时必须逐条有终态

`run()`必须确保到达的每个请求**恰好一个**终态。常见结果：

- `completed_eos`：模型生成 EOS。
- `completed_cap`：达到预先定义的解码步数上限。
- `queue_rejected` / `oversize_rejected`：尚未入场就被拒绝。
- `deadline_waiting` / `deadline_active`：在等待或活动期间到达期限。
- `cancelled_waiting` / `cancelled_active`：人为取消。
- `kv_error`：意外物理KV容量错误，不伪称正常完成。

所有活动终态必须释放真实KV页；缓存pin只在最终flush时释放。调度错误、非法身份和缓存损坏应抛异常，让整个场景显式失败。

`queue_peak`、`peak_active` 和物理页峰值必须分别计算；不能只拿一个请求计数替代字节占用。

## 7. 分位数口径与“公平性”不要过度解释

队列等待是 `admit_tick - arrival_tick`，经验P95定义为**nearest-rank**：升序后第 `ceil(0.95*n)` 个样本。这和插值分位数不是同一口径；空数据返回N/A。

各租户的完成率分别为 `completed/arrived`。Jain指数可以概述完成率是否接近，但它无法告诉你绝对完成率高不高：`[0.05,0.05]` 的Jain仍为1。因此必须同时打印每个租户的**到达数、完成数、拒绝/超时原因**。

本课报告中的“time to first output”同样是**逻辑tick**，绝不是实测TTFT。真实TTFT/TPOT定义依赖网络与客户端测量点。[vLLM Benchmarking](https://github.com/vllm-project/vllm/blob/main/docs/benchmarking/cli.md)

In [16]:
print('nearest-rank [1,2,3,10,11] P50=',nearest_rank([1,2,3,10,11],.5))
print('P95=',nearest_rank([1,2,3,10,11],.95))
print('Jain [0.05,0.05]:',jain_index([.05,.05]))
print('Jain [1,0]:',jain_index([1.,0.]))
print('空样本P95:',nearest_rank([],.95))

nearest-rank [1,2,3,10,11] P50= 3
P95= 11
Jain [0.05,0.05]: 1.0
Jain [1,0]: 0.5
空样本P95: None


## 8. 9 组有配对意义的负载对照

所有场景的模型权重与工作负载发生器都固定，同一对照只改明确指明的资源/策略选项。

| 场景 | 变动项 | 主要要观察的量 |
|---|---|---|
| `hot_no_cache` vs `hot_with_cache` | 缓存pin上限从0到30 | **真实prefill计算token** |
| `burst_fifo` vs `burst_tenant_rr` | 队列调度策略 | 每租户完成率、拒绝、期限到期 |
| `thrash_small_cache` vs `thrash_large_cache` | 缓存块上限3 vs 28 | 被缓存的完整块token数 |
| `physical_pressure` | 总容量仅15个块 | **capacity deferral**与物理峰值 |
| `isolation_versions` | 租户与模型版本交错 | 不能把跨范围公共文本当可共享命中 |
| `cancellation_deadline` | 取消、期限与突发到达 | 终态守恒与KV最终归零 |

**性能误区：** 缓存省掉N个prefill token，不等于TTFT提高N倍。真实调度还受内核、内存带宽、请求并发、解码和系统开销影响。本课不宣称高吞吐基准。

In [17]:
# %% 11 容量/命中/租户负载实验与固定对照

def new_environment(tok,models,config:LoadConfig):
    config.validate()
    authority=TrustedScopeAuthority({'alice':'private_alice',
        'alice_worker':'private_alice','bob':'private_bob',
        'org1':'trusted_org','org2':'trusted_org'})
    pool=PrefixKVPool(config.capacity_blocks,config.block_size,
        next(iter(models.values())).cfg,set(models),cache_capacity_blocks=config.cache_limit)
    service=PrefixCacheRunner(tok,models,pool,authority)
    return service


def scenario_specs():
    stable=make_workload('steady_hot',18)
    skew=make_workload('burst_skew',30)
    thrashed=make_workload('thrash',16)
    separated=make_workload('isolation',12)
    cancelled=make_workload('cancel',18)
    pressure=[TrafficRequest(sid=f'prs_{i:03d}',arrival_tick=(i//3)*2,
            principal=('alice' if i%2==0 else 'bob'),
            body=f'Hello world! unique-{i:02d}',patience_ticks=12,decode_steps=2)
            for i in range(18)]
    validate_workload(pressure)
    return [
      ('hot_no_cache',LoadConfig(capacity_blocks=56,cache_limit=0,max_active=2,
              per_principal_active=2,queue_limit=10,max_ticks=90),stable),
      ('hot_with_cache',LoadConfig(capacity_blocks=56,cache_limit=30,max_active=2,
              per_principal_active=2,queue_limit=10,max_ticks=90),stable),
      ('burst_fifo',LoadConfig(capacity_blocks=27,cache_limit=12,max_active=2,
              per_principal_active=1,queue_limit=9,max_ticks=40,policy='fifo'),skew),
      ('burst_tenant_rr',LoadConfig(capacity_blocks=27,cache_limit=12,max_active=2,
              per_principal_active=1,queue_limit=9,max_ticks=40,policy='tenant_rr'),skew),
      ('thrash_small_cache',LoadConfig(capacity_blocks=52,cache_limit=3,max_active=2,
              per_principal_active=2,queue_limit=10,max_ticks=80),thrashed),
      ('thrash_large_cache',LoadConfig(capacity_blocks=52,cache_limit=28,max_active=2,
              per_principal_active=2,queue_limit=10,max_ticks=80),thrashed),
      ('physical_pressure',LoadConfig(capacity_blocks=15,cache_limit=6,max_active=3,
              per_principal_active=2,queue_limit=10,max_ticks=60),pressure),
      ('isolation_versions',LoadConfig(capacity_blocks=52,cache_limit=28,max_active=2,
              per_principal_active=2,queue_limit=10,max_ticks=70),separated),
      ('cancellation_deadline',LoadConfig(capacity_blocks=27,cache_limit=10,max_active=2,
              per_principal_active=1,queue_limit=8,max_ticks=40),cancelled),
    ]


def run_scenario(tok,models,config:LoadConfig,items:list[TrafficRequest]):
    torch.set_num_threads(1)
    service=new_environment(tok,models,config)
    scheduler=LoadScheduler(service,config,items)
    report=scheduler.run()
    return report


def suite_summary(cases:dict[str,dict]):
    require('hot_no_cache' in cases and 'hot_with_cache' in cases,'需要冷/热缓存对照')
    cold=cases['hot_no_cache']['summary'];warm=cases['hot_with_cache']['summary']
    require(cold['prefill_tokens_reused']==0, 'cache_limit=0不允许复用')
    require(cold['prefill_tokens_total']==warm['prefill_tokens_total'],
            '冷热缓存对照必须服务同等token请求')
    require(warm['prefill_tokens_computed']<cold['prefill_tokens_computed'],
            '热缓存应减少真实prefill计算量')
    require(all(case['final_flush']['all_bytes_zero'] for case in cases.values()),
            '每个场景必须释放所有物理页')
    table=[]
    for name,case in cases.items():
        s=case['summary']
        table.append({'scenario':name,'arrived':s['requests'],'admitted':s['admitted'],
          'completed':s['completed'],'queue_rejected':s['queue_rejections'],
          'prefill_total':s['prefill_tokens_total'],
          'prefill_computed':s['prefill_tokens_computed'],
          'hit_token_fraction':s['prompt_token_hit_fraction'],
          'eligible_hit_fraction':s['eligible_full_block_hit_fraction'],
          'peak_physical_blocks':s['physical_live_blocks_peak'],
          'wait_p95_logical_ticks':s['logical_wait_ticks_p95'],
          'capacity_deferrals':s['capacity_deferral_events']})
    return {'scenarios':table,
       'saved_prefill_tokens_cold_vs_warm':cold['prefill_tokens_computed']-warm['prefill_tokens_computed'],
       'warning':'无TTFT/TPOT/GPU吞吐或真实并发时延结论；相同请求录入、真实CPU KV计算与逻辑调度对照'}

In [18]:
# 小规模预演，实际跑真正的CPU模型并验证输出一致。
tiny_no_cache=run_scenario(tok,models,
    LoadConfig(capacity_blocks=48,cache_limit=0,max_active=1,per_principal_active=1,max_ticks=40),
    make_workload('steady_hot',4))
tiny_with_cache=run_scenario(tok,models,
    LoadConfig(capacity_blocks=48,cache_limit=20,max_active=1,per_principal_active=1,max_ticks=40),
    make_workload('steady_hot',4))
print('无缓存实际计算prefill tokens:',tiny_no_cache['summary']['prefill_tokens_computed'])
print('热缓存实际计算prefill tokens:',tiny_with_cache['summary']['prefill_tokens_computed'])
print('全上下文logits误差:',tiny_with_cache['summary']['max_logit_abs_delta'])
print('最终活跃物理页:',tiny_with_cache['final_flush']['live_blocks'])

无缓存实际计算prefill tokens: 155
热缓存实际计算prefill tokens: 55
全上下文logits误差: 4.76837158203125e-07
最终活跃物理页: 0


## 9. 完整运行并保存所有统计

下面默认执行9组离线真实KV负载，生成一次新的时间戳目录。**每组场景结束后必须全部终止会话、释放缓存pin并检查所有物理KV字节归零**。

脚本将保存每个场景的**原始队列轨迹、每个请求的终态理由和实测prefill复用token数**，以及总结与可信权重重载。保存的是合成请求结构及数值；报告不输出真实个人提示，也不保存权威cache salt。

测试结果只属于这个离线CPU小模型。真实集群的负载、TTFT、TPOT、吞吐与公平性需要独立的端到端测量。

In [19]:
def save_models(path:Path,tok,models,seed):
    payload={'version':VERSION,'seed':seed,
        'cfg':asdict(next(iter(models.values())).cfg),'tokenizer':tok.contract(),
        'hashes':{v:state_hash(m.state_dict()) for v,m in models.items()},
        'states':{v:{k:t.detach().cpu().clone() for k,t in m.state_dict().items()}
                  for v,m in models.items()}}
    torch.save(payload,path)


def load_models(path:Path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'检查点版本不匹配')
    tok,models=build_models(blob['seed'])
    require(blob['cfg']==asdict(next(iter(models.values())).cfg) and
            blob['tokenizer']==tok.contract(),'模型配置或tokenizer契约不符')
    require(set(models)==set(blob['states'])==set(blob['hashes']),'模型版本集合不匹配')
    for name,model in models.items():
        require(state_hash(blob['states'][name])==blob['hashes'][name], '权重字节摘要不一致')
        model.load_state_dict(blob['states'][name],strict=True)
        require(state_hash(model.state_dict())==blob['hashes'][name], '加载后权重不一致')
        model.eval().requires_grad_(False)
    return tok,models,blob['seed']


def reload_run(folder:Path):
    folder=Path(folder)
    tok,models,seed=load_models(folder/'models.pt')
    saved=json.loads((folder/'hot_with_cache.json').read_text(encoding='utf-8'))
    reference=next((cfg,rows) for name,cfg,rows in scenario_specs() if name=='hot_with_cache')
    report=run_scenario(tok,models,*reference)
    require(report==saved,'独立模型/空KV池复跑与保存的热缓存报告不一致')
    return {'model_states_verified':True,'replayed_scenario':'hot_with_cache',
            'exact_saved_report_equal':True,'not_live_KV_checkpoint':True}


def fixture_demo(output_root:Path=Path('lesson_75_outputs'),seed:int=7501):
    torch.set_num_threads(1)
    require(type(seed) is int and seed>=0,'seed非法')
    folder=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    folder.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'python':platform.python_version(),'torch':str(torch.__version__),
              'device':'cpu','seed':seed,'status':'running','sources':SOURCES,
              'boundaries':['CPU_real_KV_prefill_decode_and_full_logits_equality',
                'logical_tick_simulation_not_real_latency','no_real_TTFT_TPOT_GPU_goodput_claim',
                'serial_deterministic_scheduler_not_threadsafe_production',
                'no_real_network_or_auth_gateway','synthetic_requests_only',
                'cached_logical_reference_not_real_workload_telemetry',
                'replay_from_weights_not_live_cache_checkpoint']}
    write_json(folder/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        save_models(folder/'models.pt',tok,models,seed)
        write_json(folder/'weights.json',{'version':VERSION,'seed':seed,
              'tokenizer_hash':digest(tok.contract()),
              'model_hashes':{k:state_hash(v.state_dict()) for k,v in models.items()}})
        cases={}
        for name,cfg,traffic in scenario_specs():
            print('离线负载:',name,flush=True)
            report=run_scenario(tok,models,cfg,traffic)
            cases[name]=report
            write_json(folder/(name+'.json'),report)
        summary=suite_summary(cases)
        write_json(folder/'summary.json',summary)
        reload=reload_run(folder)
        write_json(folder/'reload.json',reload)
        manifest['status']='completed';write_json(folder/'manifest.json',manifest)
        print('缓存关闭的prefill计算:',cases['hot_no_cache']['summary']['prefill_tokens_computed'])
        print('缓存开启的prefill计算:',cases['hot_with_cache']['summary']['prefill_tokens_computed'])
        print('已完成请求的终态/容量已审计；最后所有场景的KV归零。')
        print('输出目录:',folder)
        return folder
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(folder/'manifest.json',manifest)
        raise

In [20]:
RUN_COMPLETE_DEMO = True
RUN_DIR = None
if RUN_COMPLETE_DEMO:
    RUN_DIR = fixture_demo(Path('lesson_75_outputs'),7501)
else:
    print('已跳过完整演练，没有制造模拟结果或版本文件。')

离线负载: hot_no_cache


离线负载: hot_with_cache


离线负载: burst_fifo


离线负载: burst_tenant_rr


离线负载: thrash_small_cache


离线负载: thrash_large_cache


离线负载: physical_pressure


离线负载: isolation_versions


离线负载: cancellation_deadline


缓存关闭的prefill计算: 696
缓存开启的prefill计算: 80
已完成请求的终态/容量已审计；最后所有场景的KV归零。
输出目录: lesson_75_outputs/run_20261104_143839_663809


## 10. 本次主要数字：真实计算量，不是GPU耗时

下面表格读取刚刚由本机脚本实际保存的报告，绝不预先写死结果。

在制作环境的默认CPU运行中：同一个热前缀负载，关闭缓存需要**696个**实际新prefill token，开启缓存只需**80个**。本次命中的原始prompt token比例约88.5%；这不是延迟的百分比改善。

小缓存与大缓存的冷热混合负载则分别需要446和94个prefill新token。它们服务的合成请求一致，变化是cache pin可保留的数量。

在突发偏斜流量中，FIFO的总完成数是3/30、轮转是7/30；其中bob分别完成1/5和5/5。本课不能据此声称RR对任何负载总更优；它只是给出一个可审计的队头阻塞/租户配额反例。

In [21]:
if RUN_DIR is not None:
    table = json.loads((RUN_DIR/'summary.json').read_text(encoding='utf-8'))['scenarios']
    print('场景                    admitted/completed  复用率(token)  prefill新算  容量等待  p95等待tick')
    for x in table:
        hit = 'N/A' if x['hit_token_fraction'] is None else f"{100*x['hit_token_fraction']:.1f}%"
        print(f"{x['scenario']:<25s} {x['admitted']:>2}/{x['completed']:<2}           "
              f"{hit:>6}          {x['prefill_computed']:>4}        "
              f"{x['capacity_deferrals']:>3}        {x['wait_p95_logical_ticks']}")
else:
    print('完整演练已跳过，未使用预设报告。')

场景                    admitted/completed  复用率(token)  prefill新算  容量等待  p95等待tick
hot_no_cache              18/18             0.0%           696          0        0
hot_with_cache            18/18            88.5%            80          0        0
burst_fifo                11/3             35.4%           270          0        4
burst_tenant_rr           12/7             54.4%           208          0        4
thrash_small_cache        16/16            28.8%           446          0        0
thrash_large_cache        16/16            85.0%            94          0        0
physical_pressure         11/7              5.3%           427         40        10
isolation_versions        12/12            71.1%           132          0        0
cancellation_deadline     11/3             37.4%           275          0        4


## 11. 负载分布、公平性、队列终态要一起看

比较两种策略时，必须同时查看**负载提供给系统的请求数、实际准入数、完成数和队列拒绝**。只统计已准入请求，会隐藏拥塞下被拒绝的用户。

本课每个租户都有独立最大活动数。租户轮转只在这份逻辑调度器内定义，并不表示任何系统的实际调度代码都如此实现。

`capacity_deferral_events` 是每次因物理容量不足而不能入场的尝试次数，同一请求可能被多次延期，**不是被拒绝的请求数**。压力场景需同时看等待终态和最终内存归零。

In [22]:
if RUN_DIR is not None:
    for name in ['burst_fifo','burst_tenant_rr','physical_pressure','cancellation_deadline']:
        case=json.loads((RUN_DIR/(name+'.json')).read_text(encoding='utf-8'))
        s=case['summary']
        print('\n',name,'全部终态=',s['terminal_counts'],
              '容量预检延后事件=',s['capacity_deferral_events'])
        for principal,v in s['per_principal'].items():
            print(' ',principal,':',v['completed'],'/',v['arrived'],
                  '原因=',v['reasons'])


 burst_fifo 全部终态= {'completed_cap': 3, 'deadline_active': 8, 'deadline_waiting': 10, 'queue_rejected': 9} 容量预检延后事件= 0
  alice : 2 / 25 原因= {'completed_cap': 2, 'deadline_active': 4, 'deadline_waiting': 10, 'queue_rejected': 9}
  bob : 1 / 5 原因= {'completed_cap': 1, 'deadline_active': 4}

 burst_tenant_rr 全部终态= {'completed_cap': 7, 'deadline_active': 5, 'deadline_waiting': 11, 'queue_rejected': 7} 容量预检延后事件= 0
  alice : 2 / 25 原因= {'completed_cap': 2, 'deadline_active': 5, 'deadline_waiting': 11, 'queue_rejected': 7}
  bob : 5 / 5 原因= {'completed_cap': 5}

 physical_pressure 全部终态= {'completed_cap': 7, 'deadline_active': 4, 'deadline_waiting': 4, 'queue_rejected': 3} 容量预检延后事件= 40
  alice : 4 / 9 原因= {'completed_cap': 4, 'deadline_active': 1, 'deadline_waiting': 2, 'queue_rejected': 2}
  bob : 3 / 9 原因= {'completed_cap': 3, 'deadline_active': 3, 'deadline_waiting': 2, 'queue_rejected': 1}

 cancellation_deadline 全部终态= {'cancelled_active': 1, 'cancelled_waiting': 1, 'completed_cap': 3, 'de

## 12. 物理预算要分别报告“预分配”和“实际活跃”

KV理论公式（这份模型使用同头数的K/V，不含GQA）：

\[
\mathrm{bytes/token}=2\times L\times H\times D_{head}\times\mathrm{dtype\_bytes}
\]

物理块字节数等于上式乘`block_size`。池在创建时**一次性预分配**了整个Tensor，所以 `physical_storage_bytes` 不会随请求减少；而 `live_reserved_bytes` 和 cache pins 随场景变化。

缓存条目还保存哈希、token元数据和末端logits，这些Python对象内存**没有计入KV块张量字节预算**；不能把此理论账本说成整个进程的RSS或GPU峰值显存。

In [23]:
if RUN_DIR is not None:
    case=json.loads((RUN_DIR/'physical_pressure.json').read_text(encoding='utf-8'))['summary']
    print('物理容量block:',case['physical_capacity_blocks'])
    print('每block字节数:',case['kv_bytes_per_block'])
    print('预分配的KV tensor字节:',case['kv_pool_bytes_preallocated'])
    print('实际活跃物理block峰值:',case['physical_live_blocks_peak'])
    print('容量等待事件数:',case['capacity_deferral_events'])
    print('场景结束后物理字节清零:',json.loads((RUN_DIR/'physical_pressure.json').read_text())['final_flush']['all_bytes_zero'])

物理容量block: 15
每block字节数: 2048
预分配的KV tensor字节: 30720
实际活跃物理block峰值: 11
容量等待事件数: 40
场景结束后物理字节清零: True


## 13. 检查：物理计算正确，还要能发现“很像合理指标”的错误

本课的自动检查覆盖：命中分母、空分母、nearest-rank、Jain、来源身份、完整块、真实KV和full-forward一致、跨租户隔离、缓存损坏拒绝、队列溢出、取消与期限、单请求物理容量超限、只读预检不动LRU、真实容量延期、终态守恒和KV全归零。

容量不会被超过，是**这份同步模型的账本性质**；不代表现实中已经验证了多线程锁、GPU allocator、网络鉴权或系统延迟。

很重要：**不能把批量故障注入超时记作通过**。若改错一个关键条件，请单项执行检查确认它真的检测到了相应错误。

In [24]:
# %% 12 自动检查、数学练习与出错路径
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,ok):
        if not bool(ok):raise AssertionError(name)
        self.names.append(name)
    def near(self,name,a,b,eps=1e-10):
        self.ok(name,abs(float(a)-float(b))<=eps)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError):self.names.append(name);return
        raise AssertionError(name+'：应该拒绝但未拒绝')


def run_checks()->dict:
    torch.set_num_threads(1)
    c=Checks()
    c.near('1 命中token定义',metric_ratio(16,64),.25)
    c.near('2 请求命中比例',metric_ratio(3,4),.75)
    c.ok('3 零分母返回None',metric_ratio(0,0) is None)
    c.rejects('4 不允许分子超过分母',lambda:metric_ratio(2,1))
    c.rejects('5 不允许负计数',lambda:metric_ratio(-1,1))
    c.ok('6 最近秩空样本',nearest_rank([],.95) is None)
    c.ok('7 最近秩P50',[nearest_rank([9,1,3,6],.5),nearest_rank([9,1,3,6],.95)]==[3,9])
    c.rejects('8 分位数p非法',lambda:nearest_rank([1],1.1))
    c.near('9 Jain完全均衡',jain_index([.5,.5]),1.)
    c.near('10 Jain极端倾斜',jain_index([1.,0.]),.5)
    c.ok('11 Jain全零不定义',jain_index([0.,0.]) is None)
    c.rejects('12 Jain拒绝负数',lambda:jain_index([1.,-1.]))
    c.rejects('13 请求ID重复拒绝',lambda:validate_workload(make_workload('steady_hot',2)*2))
    c.rejects('14 工作负载乱序拒绝',lambda:validate_workload(list(reversed(make_workload('steady_hot',3)))))
    c.rejects('15 无效租户拒绝',lambda:TrafficRequest('a',0,'mallory','x').validate())
    c.rejects('16 负arrival拒绝',lambda:TrafficRequest('a',-1,'alice','x').validate())
    c.rejects('17 0patience拒绝',lambda:TrafficRequest('a',0,'alice','x',patience_ticks=0).validate())
    c.rejects('18 decode预算越界',lambda:TrafficRequest('a',0,'alice','x',decode_steps=4).validate())
    c.rejects('19 非法配置cache>capacity',lambda:LoadConfig(capacity_blocks=2,cache_limit=3).validate())
    c.rejects('20 非法并发上限',lambda:LoadConfig(max_active=1,per_principal_active=2).validate())
    c.rejects('21 未知调度策略拒绝',lambda:LoadConfig(policy='magic').validate())
    c.ok('22 请求形状稳定',len(make_workload('steady_hot',8))==8)
    c.ok('23 租户负载记录不共享身份',make_workload('isolation',4)[0].principal!=make_workload('isolation',4)[1].principal)
    c.ok('24 突发流量同tick多个到达',make_workload('burst_skew',12)[0].arrival_tick==make_workload('burst_skew',12)[5].arrival_tick)
    c.ok('25 取消流量包含真实cancel',any(x.cancel_after_ticks is not None for x in make_workload('cancel',8)))
    tok,models=build_models(7501)
    no_cache=LoadConfig(capacity_blocks=48,cache_limit=0,max_active=1,
                  per_principal_active=1,queue_limit=8,max_ticks=40)
    warm=LoadConfig(capacity_blocks=48,cache_limit=24,max_active=1,
                  per_principal_active=1,queue_limit=8,max_ticks=40)
    sample=make_workload('steady_hot',4)
    cold=run_scenario(tok,models,no_cache,sample)
    hot=run_scenario(tok,models,warm,sample)
    c.ok('26 禁用缓存的token命中0',cold['summary']['prefill_tokens_reused']==0)
    c.ok('27 缓存减少真实计算',hot['summary']['prefill_tokens_computed']<cold['summary']['prefill_tokens_computed'])
    c.ok('28 同负载admitted相等',hot['summary']['admitted']==cold['summary']['admitted'])
    c.ok('29 命中率分母相同',hot['summary']['prefill_tokens_total']==cold['summary']['prefill_tokens_total'])
    c.ok('30 容量硬上限遵守',all(s['physical_live_blocks_peak']<=s['physical_capacity_blocks']
              for s in (cold['summary'],hot['summary'])))
    c.ok('31 结束后KV字节归零',hot['final_flush']['all_bytes_zero'])
    c.ok('32 输出全长概率一致',hot['summary']['max_logit_abs_delta']<2e-5)
    c.ok('33 prompt账目守恒',hot['summary']['prefill_tokens_computed']+hot['summary']['prefill_tokens_reused']==hot['summary']['prefill_tokens_total'])
    c.ok('34 物理页实际占用有峰值',hot['summary']['physical_live_blocks_peak']>0)
    c.ok('35 eligible比例不低于raw比例',hot['summary']['eligible_full_block_hit_fraction']>=hot['summary']['prompt_token_hit_fraction'])
    c.ok('36 所有终态各一次',len({r['sid'] for r in hot['terminal']})==len(sample))
    c.ok('37 热缓存全量正确完成',hot['summary']['completed']==len(sample))
    c.ok('38 缓存对照不产生幽灵OOM',hot['summary']['kv_errors']==0)
    c.ok('39 cache_limit0无缓存保留',cold['summary']['cache_pins_before_flush']==0)
    c.ok('40 热缓存保留独立pins',hot['summary']['cache_pins_before_flush']>0)
    c.ok('41 热缓存更高请求命中比例',hot['summary']['request_cache_hit_fraction']>0)
    c.ok('42 queue等待逻辑tick非负',hot['summary']['logical_wait_ticks_p95']>=0)
    c.ok('43 无用户原始正文存报告',all('body' not in r for r in hot['terminal']))
    c.ok('44 模型实际计算过token',hot['summary']['actually_computed_model_tokens']>0)
    c.ok('45 清空后cache_pins归0',hot['final_flush']['cache_pins']==0)
    c.ok('46 round robin不超过单租户额度',hot['summary']['active_peak']<=1)
    # 真正不允许未知/不隔离的主体，也不容忍缓存损坏变成miss。
    service=new_environment(tok,models,LoadConfig(capacity_blocks=44,cache_limit=24,max_active=2))
    auth=service.authority
    r1=CacheRequest('auditA','alice','v0001',(('system','Reply briefly.'),('user','Hello world! repeat')))
    r2=CacheRequest('auditB','bob','v0001',r1.messages)
    r3=CacheRequest('auditW','alice_worker','v0001',r1.messages)
    a=service.admit(r1)
    c.ok('47 首次前缀冷启动',a['hit_tokens']==0)
    service.close('auditA')
    b=service.admit(r2)
    c.ok('48 跨租户不能命中',b['hit_tokens']==0)
    service.close('auditB')
    w=service.admit(r3)
    c.ok('49 同共享组可命中',w['hit_tokens']>0)
    service.close('auditW')
    c.rejects('50 未知principal拒绝',lambda:service.admit(CacheRequest('bad','mallory','v0001',r1.messages)))
    require(bool(service.pool.cache),'测试需要缓存块')
    e=next(iter(service.pool.cache.values()))
    with torch.no_grad():service.pool.data[e.physical_page,0,0,0,0,0]+=1
    c.rejects('51 破坏真实KV不允许透明命中',lambda:service.pool.lookup(e.identity,list(e.prefix_tokens)))
    # 修复测试污染后，始终必须全量回收。
    service.pool.invalidate();flushed=service.final_flush()
    c.ok('52 非法缓存被显式失效后全部释放',flushed['live_blocks']==0)
    # 无论队列满还是被动取消，全部请求都必须有理由，不得静默丢失。
    burst=run_scenario(tok,models,
             LoadConfig(capacity_blocks=28,cache_limit=8,max_active=2,per_principal_active=1,
                 queue_limit=4,max_ticks=40),make_workload('burst_skew',12))
    c.ok('53 队列溢出有明确拒绝',burst['summary']['queue_rejections']>0)
    c.ok('54 所有入场请求都有终态',len(burst['terminal'])==burst['summary']['requests'])
    c.ok('55 物理页上限守恒',burst['summary']['physical_live_blocks_peak']<=28)
    c.ok('56 突发流量到达时队列确实打满上限',burst['summary']['queue_peak']>=4)
    c.ok('56b 排队峰值不能越过配置上限',burst['summary']['queue_peak']<=4)
    c.ok('57 被拒绝请求不计prefix分母',burst['summary']['prefill_tokens_total']==sum(x['prompt_tokens'] for x in burst['admitted']))
    c.ok('58 真实模型logits核对',burst['summary']['max_logit_abs_delta']<2e-5)
    c.ok('59 终态包含拒绝理由',any(x['reason']=='queue_rejected' for x in burst['terminal']))
    c.ok('60 只保存逻辑ticks不伪称毫秒',all('ms' not in k for k in burst['summary']))
    cancellation=run_scenario(tok,models,
        LoadConfig(capacity_blocks=28,cache_limit=10,max_active=2,per_principal_active=1,
                 queue_limit=6,max_ticks=30),make_workload('cancel',12))
    c.ok('61 主动取消事件真实发生',any('cancelled' in x['reason'] for x in cancellation['terminal']))
    c.ok('62 已接受会话最终为0',cancellation['final_flush']['live_blocks']==0)
    c.ok('63 队列长度从未超过限制',all(x['queued']<=6 for x in cancellation['trace']))
    c.ok('64 运行中并发不超过2',all(x['active']<=2 for x in cancellation['trace']))
    c.ok('65 每tick终态数单调',all(a['terminal']<=b['terminal'] for a,b in zip(cancellation['trace'],cancellation['trace'][1:])))
    c.ok('66 每个会话仅一个终态',len({x['sid'] for x in cancellation['terminal']})==12)
    c.ok('67 结束后零活动引用',cancellation['final_flush']['cache_pins']==0)
    # 极小池预检无法容纳单条完整请求，不能在队列里无限旋转。
    too_small=run_scenario(tok,models,
        LoadConfig(capacity_blocks=2,cache_limit=1,max_active=1,
                   per_principal_active=1,queue_limit=5,max_ticks=20),make_workload('steady_hot',3))
    c.ok('68 单条提示超过物理容量必须终止',too_small['summary']['terminal_counts'].get('oversize_rejected',0)==3)
    c.ok('69 拒绝不会尝试物理KV计算',too_small['summary']['prefill_tokens_total']==0)
    c.ok('70 空评测无分母返回None',too_small['summary']['prompt_token_hit_fraction'] is None)
    c.ok('71 并发物理页最终为空',too_small['final_flush']['live_blocks']==0)
    # 前缀预检不能改变cache的LRU：一个查询未被准入就不应伪造命中。
    svc=new_environment(tok,models,LoadConfig(capacity_blocks=40,cache_limit=24))
    sched=LoadScheduler(svc,LoadConfig(capacity_blocks=40,cache_limit=24),make_workload('steady_hot',2))
    first_plan=sched._plan(sched.items[0].sid)
    before_clock=svc.pool.clock
    c.ok('72 只读规划不触碰LRU逻辑时钟',svc.pool.clock==before_clock)
    c.ok('73 未计算的前缀估计0命中',first_plan['hit_blocks']==0)
    c.ok('74 预检包含decode预留',first_plan['required_new']>=first_plan['prompt_blocks'])
    c.ok('75 可用物理页不大于capacity',first_plan['available_new']<=svc.pool.capacity_blocks)
    c.ok('76 缓存对象没有多余活动会话',not svc.pool.sessions)
    c.ok('77 真实池能完整零化',svc.final_flush()['live_blocks']==0)
    # 校验实际物理压力、热/冷命中、租户RR，而不是只测数值函数。
    pressure_items=[TrafficRequest(sid=f'cpr_{i}',arrival_tick=(i//3)*2,
             principal=('alice' if i%2==0 else 'bob'),
             body=f'Hello world! unique-{i:02d}',patience_ticks=12)
             for i in range(9)]
    pressure_cfg=LoadConfig(capacity_blocks=15,cache_limit=6,max_active=3,
                 per_principal_active=2,queue_limit=8,max_ticks=40)
    pressure=run_scenario(tok,models,pressure_cfg,pressure_items)
    ps=pressure['summary']
    c.ok('78 真正容量不足出现排队预检',ps['capacity_deferral_events']>0)
    c.ok('79 压力场景不突破物理页',ps['physical_live_blocks_peak']<=15)
    c.ok('80 容量不足不被记为成功',ps['completed']<=ps['admitted']<=ps['requests'])
    c.ok('81 压力场景终态不遗漏',sum(ps['terminal_counts'].values())==ps['requests'])
    c.ok('82 真实KV输出在容量压力下仍一致',ps['max_logit_abs_delta']<2e-5)
    c.ok('83 压力结束后物理KV清空',pressure['final_flush']['all_bytes_zero'])
    c.ok('84 不将拒绝请求统计到prompt分母',sum(x['prompt_tokens'] for x in pressure['admitted'])==ps['prefill_tokens_total'])
    c.ok('85 空闲缓存和活动页总峰值不越界',all(r['physical_live_blocks']<=15 for r in pressure['trace']))
    c.ok('86 每tick的到达数不回退',all(a['arrived']<=b['arrived'] for a,b in zip(pressure['trace'],pressure['trace'][1:])))
    c.ok('87 每个到达请求恰好出现一次终态',len({r['sid'] for r in pressure['terminal']})==len(pressure_items))
    c.rejects('88 仿真窗口不能短于最后arrival',lambda:LoadScheduler(
          new_environment(tok,models,LoadConfig(max_ticks=3)),
          LoadConfig(max_ticks=3),make_workload('steady_hot',2)))
    c.ok('89 Jain只可描述完成比例不是时延',ps['fairness_jain_completion_fraction'] is None or
           0.5<=ps['fairness_jain_completion_fraction']<=1)
    c.ok('90 cache block查询比例与完整块比例一致',
           ps['cache_block_query_hit_fraction']==ps['eligible_full_block_hit_fraction'])
    c.ok('91 活跃队列峰值不超过配置',ps['queue_peak']<=pressure_cfg.queue_limit)
    c.ok('92 所有模型权重在负载前后不变化',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    return {'passed':len(c.names),'names':c.names,
            'scope':'真实CPU KV tensor+确定性负载，不是GPU吞吐、延迟、生产并发或安全证明'}

In [25]:
checks=run_checks()
print('真实CPU KV和负载检查通过:',checks['passed'],'项')
print('测试范围:',checks['scope'])
if RUN_DIR is not None:
    write_json(RUN_DIR/'checks.json',checks)

真实CPU KV和负载检查通过: 93 项
测试范围: 真实CPU KV tensor+确定性负载，不是GPU吞吐、延迟、生产并发或安全证明


## 14. 保存与从全新空KV池重放

演练文件结构：

| 文件 | 用途 |
|---|---|
| `models.pt` / `weights.json` | 两个冻结LoRA版本、结构与内容摘要 |
| `hot_no_cache.json` / `hot_with_cache.json` | 同负载的冷/热缓存真实prefill报告 |
| `burst_fifo.json` / `burst_tenant_rr.json` | 租户调度与排队终态 |
| `thrash_small_cache.json` / `thrash_large_cache.json` | 缓存容量压力 |
| `physical_pressure.json` | 真正物理页预算紧张，出现容量等待 |
| `isolation_versions.json` / `cancellation_deadline.json` | 可信隔离与完整终态 |
| `summary.json` / `reload.json` / `checks.json` | 汇总、只读重放、检查 |

重放**只加载可信本地权重并新建空KV池与新的服务端salt**，核对热缓存场景的全部统计。没有跨进程恢复仍在使用的物理KV页，也不会读取不可信PyTorch pickle；本课使用 `weights_only=True`。

In [26]:
if RUN_DIR is not None:
    print(json.dumps(reload_run(RUN_DIR),ensure_ascii=False,indent=2))
else:
    print('演练未执行，没有检查点可供重放。')

{
  "model_states_verified": true,
  "replayed_scenario": "hot_with_cache",
  "exact_saved_report_equal": true,
  "not_live_KV_checkpoint": true
}


## 15. 独立练习 A：缓存token命中率

实现 `hit_rate_ex(hit_tokens,total_tokens,eligible_tokens=None)`。返回原始prompt token命中率而不是eligible分母；分母为0返回None，非法计数明确拒绝。

小题：复用15个token，在已入场的52个prompt token中；可缓存完整块的token只有48个。为什么15/52与15/48都合理，但必须分别命名？

In [27]:
def exercise_a(fn):
    require(fn(15,52,48)==15/52,'分子应是复用token，分母是全部 admitted prompt token')
    require(fn(0,0) is None,'空分母不能装作0%或100%')
    try:fn(7,5)
    except (LessonError,ValueError):pass
    else:raise AssertionError('不守恒计数没有拒绝')
    return {'exercise':'A','passed':True}


def exercise_b(fn):
    require(fn([1,2,3,10,11],.95)==11,'nearest rank P95错误')
    require(fn([1,2,3,10,11],.5)==3,'nearest rank P50错误')
    require(fn([],.95) is None,'空样本必须None')
    return {'exercise':'B','passed':True}


def exercise_c(fn):
    require(fn(2,8,1,6)==False,'需要两块但只有一空闲页，不应入场')
    require(fn(3,8,0,6)==True,'三块命中时只需五块，可用6块应入场')
    require(fn(5,10,0,5)==True,'刚好能够容纳应允许')
    try:fn(3,2,1,5)
    except (LessonError,ValueError):pass
    else:raise AssertionError('命中块数不能超过总块数')
    return {'exercise':'C','passed':True}

In [28]:
def hit_rate_ex(hit_tokens,total_tokens,eligible_tokens=None):
    raise NotImplementedError('请独立实现命中率及分母校验')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(exercise_a(hit_rate_ex))

## 16. 独立练习 B：P95经验分位数

实现 `wait_p95_ex(values,p)`。使用nearest-rank算法，而不是 `numpy.percentile` 的默认插值方法。空数组返回None；p必须位于(0,1]。不要把这些逻辑tick与毫秒混同。

In [29]:
def wait_p95_ex(values,p):
    raise NotImplementedError('请独立实现nearest-rank分位数')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(exercise_b(wait_p95_ex))

## 17. 独立练习 C：简化的物理页准入预算

实现 `can_admit_ex(hit_blocks,total_prompt_blocks,other_reserved_blocks,available_blocks)`。

只计算：新增prompt块数＋已知其他预留块数 是否不超过 可用空闲/可逐出块数。`hit_blocks`不可以超过提示总块数。

**这只是单步预算的数学练习**；完整调度器还检查租户额度、活动槽、身份、未来decode块和不允许逐出的活动引用。

In [30]:
def can_admit_ex(hit_blocks,total_prompt_blocks,other_reserved_blocks,available_blocks):
    raise NotImplementedError('请独立实现block预检')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(exercise_c(can_admit_ex))

## 18. 参考答案（完成练习后再看）

以下参考实现使用独立名字，不覆盖你的练习函数，也不会自动把练习开关改为True。

In [31]:
def answer_a(hit_tokens, total_tokens, eligible_tokens=None):
    require(type(hit_tokens) is int and type(total_tokens) is int and 0<=hit_tokens<=total_tokens,
            'token计数非法')
    if eligible_tokens is not None:
        require(type(eligible_tokens) is int and hit_tokens<=eligible_tokens<=total_tokens,
                'eligible分母错误')
    return hit_tokens/total_tokens if total_tokens else None


def answer_b(values,p):
    require(0<p<=1 and math.isfinite(p),'p错误')
    if not values:return None
    require(all(type(v) is int and v>=0 for v in values),'tick非法')
    return sorted(values)[math.ceil(len(values)*p)-1]


def answer_c(hit_blocks,total_prompt_blocks,other_reserved_blocks,available_blocks):
    require(all(type(x) is int and x>=0 for x in
                (hit_blocks,total_prompt_blocks,other_reserved_blocks,available_blocks)) and
            hit_blocks<=total_prompt_blocks, '容量预检参数非法')
    return total_prompt_blocks-hit_blocks+other_reserved_blocks<=available_blocks

In [32]:
print('练习A参考验收:',exercise_a(answer_a))
print('练习B参考验收:',exercise_b(answer_b))
print('练习C参考验收:',exercise_c(answer_c))

练习A参考验收: {'exercise': 'A', 'passed': True}
练习B参考验收: {'exercise': 'B', 'passed': True}
练习C参考验收: {'exercise': 'C', 'passed': True}


## 19. 课堂验收：5条正确的归因链

1. 为什么相同负载从696个新prefill token降到80个，**不能**写成“TTFT减少88.5%”？
2. 为什么小缓存与大缓存要报告**完整块可缓存分母**，不能只看请求hit/not hit？
3. 为什么队列满的拒绝和物理KV容量延期，要用不同计数器？
4. 为什么相同的文本由不同租户/LoRA版本提交，不能直接复用同一物理KV页？
5. 为什么一个调度器的Jain指数接近1，也可能表示“两位租户都几乎没有成功”？

**尚未实现：** 真实GPU/HTTP吞吐测试、TTFT/TPOT毫秒、生产并发线程与竞态、异步抢占、token预算动态调度、基于cost的Admission Policy、端到端压测客户端、访问时间侧信道完整防护，以及模型权重/媒体缓存跨机器迁移。

**下一课：第76课——推理 SLO、背压与过载保护。** 把逻辑容量账本接到真实客户端压测指标定义上，区分请求完成率、deadline、TTFT和尾延迟，而不是仅靠缓存命中率决定扩容。

## 参考资料

1. [vLLM Prefix Cache：完整块哈希链与复用身份](https://docs.vllm.ai/en/latest/design/prefix_caching/)
2. [vLLM Scheduler：排队、活动KV块与请求调度](https://docs.vllm.ai/en/latest/api/vllm/v1/core/sched/scheduler/)
3. [vLLM Metrics：运行/等待、KV占用、prefix命中计数](https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md)
4. [vLLM Benchmark：TTFT/TPOT的客户端测量定义](https://github.com/vllm-project/vllm/blob/main/docs/benchmarking/cli.md)
5. [PagedAttention论文](https://arxiv.org/abs/2309.06180)
6. [PyTorch 推理模式](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

本课用的是合成流量与小型冻结CPU模型；百分比都是该负载的实际计数器比值，不构成vLLM或RTX3060基准。